# Advanced Problems with Solutions — Python Property Decorators

This notebook is an **advanced practice set** centered on Python's `property` callable and decorator syntax.

## Scope

The source material covers these core ideas:

- `property(...)` creates and returns a property object.
- `@property` is decorator syntax for creating a property.
- `.setter`, `.getter`, and `.deleter` create replacement property objects with the additional accessor.
- Reusing the **same method/property name** is important when using `@name.setter`.
- A property may be read-only, write-only, read/write, or deletable.
- A property's documentation normally comes from the getter's docstring.
- "Write-only" does **not** make data truly private.

This notebook keeps those ideas at the center, then extends them into validation, invariants, inheritance, caching, reusable property factories, `__slots__`, debugging, and realistic object design.

**Best-practice convention used throughout:** public API via `obj.attribute`; internal storage via a distinct backing attribute such as `obj._attribute`.

## How to use this notebook

For every problem:

1. Read the prompt and constraints.
2. Try your own implementation before reading the solution.
3. Run the solution.
4. Run the assertions immediately after it.
5. Modify the tests to explore edge cases.

All examples use only the Python standard library.

# Part I — Mechanics and Mental Models

## Problem 1 — Prove decorator equivalence

Create two classes whose `name` attributes behave identically:

- `ManualPerson` must use `name = property(name)`.
- `DecoratedPerson` must use `@property`.
- Both properties are read-only.
- Demonstrate that the class attribute is a `property` object.

In [1]:
# Solution 1

class ManualPerson:
    def __init__(self, name):
        self._name = name

    def name(self):
        return self._name

    name = property(name)


class DecoratedPerson:
    def __init__(self, name):
        self._name = name

    @property
    def name(self):
        return self._name


m = ManualPerson("Alex")
d = DecoratedPerson("Guido")

assert m.name == "Alex"
assert d.name == "Guido"
assert isinstance(ManualPerson.__dict__["name"], property)
assert isinstance(DecoratedPerson.__dict__["name"], property)

print(type(ManualPerson.__dict__["name"]).__name__)
print(type(DecoratedPerson.__dict__["name"]).__name__)

property
property


### Key idea

Inside a class body, `@property` effectively replaces the function name with the `property` object returned by `property(function)`.

## Problem 2 — Show that `.setter(...)` returns a new property

Create a getter-only property manually, then add a setter with `.setter(...)`.

Prove that the original and returned property objects are distinct, while the new property preserves the getter and adds the setter.

In [2]:
# Solution 2

def get_score(self):
    return self._score

def set_score(self, value):
    self._score = value

getter_only = property(get_score)
read_write = getter_only.setter(set_score)

assert getter_only is not read_write
assert getter_only.fget is get_score
assert getter_only.fset is None

assert read_write.fget is get_score
assert read_write.fset is set_score

print("same object:", getter_only is read_write)
print("getter preserved:", read_write.fget is get_score)
print("setter installed:", read_write.fset is set_score)

same object: False
getter preserved: True
setter installed: True


## Problem 3 — Diagnose the naming trap

Use a getter named `name`, but deliberately define the setter function as `full_name`.

Inspect the class and prove this creates two class attributes referring to two property objects. Show that `p.name = ...` still fails while `p.full_name = ...` works.

In [3]:
# Solution 3

class NamingTrapPerson:
    def __init__(self, name):
        self._name = name

    @property
    def name(self):
        return self._name

    @name.setter
    def full_name(self, value):
        self._name = value


p = NamingTrapPerson("Alex")

assert isinstance(NamingTrapPerson.__dict__["name"], property)
assert isinstance(NamingTrapPerson.__dict__["full_name"], property)
assert NamingTrapPerson.__dict__["name"].fset is None
assert NamingTrapPerson.__dict__["full_name"].fset is not None

try:
    p.name = "Raymond"
except AttributeError as exc:
    print("p.name assignment fails:", exc)

p.full_name = "Raymond"
assert p.full_name == "Raymond"
assert p.name == "Raymond"

print("class property names:",
      [k for k, v in NamingTrapPerson.__dict__.items()
       if isinstance(v, property)])

p.name assignment fails: property 'name' of 'NamingTrapPerson' object has no setter
class property names: ['name', 'full_name']


### Best practice

When using `@name.setter` or `@name.deleter`, define the decorated function with the **same name** (`name`) unless you deliberately want another class attribute.

# Part II — Validation and Computed Properties

## Problem 4 — Validated numeric property

Implement `Temperature.celsius` with these rules:

- Accept `int` or `float`, but reject `bool`.
- Reject temperatures below absolute zero: `-273.15`.
- Store internally as `_celsius`.
- Add a read-only computed property `fahrenheit`.
- Raise `TypeError` for the wrong type and `ValueError` for an invalid numeric value.

In [4]:
# Solution 4

class Temperature:
    ABSOLUTE_ZERO_C = -273.15

    def __init__(self, celsius):
        self.celsius = celsius

    @property
    def celsius(self):
        """Temperature in degrees Celsius."""
        return self._celsius

    @celsius.setter
    def celsius(self, value):
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise TypeError("celsius must be an int or float")
        if value < self.ABSOLUTE_ZERO_C:
            raise ValueError("temperature cannot be below absolute zero")
        self._celsius = float(value)

    @property
    def fahrenheit(self):
        """Temperature converted to degrees Fahrenheit."""
        return self._celsius * 9 / 5 + 32


t = Temperature(25)
assert t.celsius == 25.0
assert t.fahrenheit == 77.0

t.celsius = 0
assert t.fahrenheit == 32.0

for bad in ("cold", None, True):
    try:
        t.celsius = bad
    except TypeError:
        pass
    else:
        raise AssertionError(f"{bad!r} should have failed")

try:
    t.celsius = -300
except ValueError as exc:
    print("expected:", exc)

print(t.celsius, t.fahrenheit)

expected: temperature cannot be below absolute zero
0.0 32.0


## Problem 5 — Normalize input in the setter

Implement `User.username` so assignment requires a string, strips whitespace, lowercases it, rejects empty results, allows only letters/digits/underscore, and stores the normalized result in `_username`.

In [5]:
# Solution 5

import re

class User:
    USERNAME_RE = re.compile(r"^[a-z0-9_]+$")

    def __init__(self, username):
        self.username = username

    @property
    def username(self):
        return self._username

    @username.setter
    def username(self, value):
        if not isinstance(value, str):
            raise TypeError("username must be a string")

        normalized = value.strip().lower()

        if not normalized:
            raise ValueError("username cannot be empty")
        if not self.USERNAME_RE.fullmatch(normalized):
            raise ValueError("username may contain only letters, digits, and underscore")

        self._username = normalized


u = User("  Ada_Lovelace  ")
assert u.username == "ada_lovelace"

u.username = "PYTHON_101"
assert u.username == "python_101"

for bad in ("", "   ", "bad-name", "name with spaces"):
    try:
        u.username = bad
    except ValueError:
        pass
    else:
        raise AssertionError(f"{bad!r} should have failed")

print(u.username)

python_101


## Problem 6 — Read-only derived properties

Implement a `Rectangle` with validated `width` and `height`, plus read-only `area`, `perimeter`, and `is_square`.

Do not store `_area` or `_perimeter`; derive them from current state.

In [6]:
# Solution 6

class Rectangle:
    def __init__(self, width, height):
        self.width = width
        self.height = height

    @staticmethod
    def _validate_dimension(name, value):
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise TypeError(f"{name} must be numeric")
        if value <= 0:
            raise ValueError(f"{name} must be > 0")
        return float(value)

    @property
    def width(self):
        return self._width

    @width.setter
    def width(self, value):
        self._width = self._validate_dimension("width", value)

    @property
    def height(self):
        return self._height

    @height.setter
    def height(self, value):
        self._height = self._validate_dimension("height", value)

    @property
    def area(self):
        return self._width * self._height

    @property
    def perimeter(self):
        return 2 * (self._width + self._height)

    @property
    def is_square(self):
        return self._width == self._height


r = Rectangle(3, 4)
assert r.area == 12
assert r.perimeter == 14
assert not r.is_square

r.width = 4
assert r.area == 16
assert r.is_square

try:
    r.area = 999
except AttributeError as exc:
    print("area is read-only:", exc)

print(r.width, r.height, r.area, r.perimeter, r.is_square)

area is read-only: property 'area' of 'Rectangle' object has no setter
4.0 4.0 16.0 16.0 True


## Problem 7 — Avoid accidental recursion in setters

A setter that executes `self.value = value` recursively calls itself. Fix it with a backing attribute and add non-negative-integer validation.

In [7]:
# Solution 7

class Counter:
    def __init__(self, value=0):
        self.value = value

    @property
    def value(self):
        return self._value

    @value.setter
    def value(self, value):
        if isinstance(value, bool) or not isinstance(value, int):
            raise TypeError("value must be an integer")
        if value < 0:
            raise ValueError("value must be non-negative")

        self._value = value


c = Counter(3)
c.value = 10
assert c.value == 10
print(c.value)

10


# Part III — Cross-Attribute Invariants

## Problem 8 — Interval invariant

Build an `Interval(start, end)` such that `start <= end` is always true. Both endpoints are mutable. Add a read-only `length`.

In [8]:
# Solution 8

class Interval:
    def __init__(self, start, end):
        if start > end:
            raise ValueError("start must be <= end")
        self._start = start
        self._end = end

    @property
    def start(self):
        return self._start

    @start.setter
    def start(self, value):
        if value > self._end:
            raise ValueError("start cannot be greater than end")
        self._start = value

    @property
    def end(self):
        return self._end

    @end.setter
    def end(self, value):
        if value < self._start:
            raise ValueError("end cannot be less than start")
        self._end = value

    @property
    def length(self):
        return self._end - self._start


interval = Interval(10, 20)
assert interval.length == 10

interval.start = 12
interval.end = 30
assert interval.length == 18

for action in (
    lambda: setattr(interval, "start", 31),
    lambda: setattr(interval, "end", 11),
):
    try:
        action()
    except ValueError as exc:
        print("expected:", exc)

print(interval.start, interval.end, interval.length)

expected: start cannot be greater than end
expected: end cannot be less than start
12 30 18


## Problem 9 — Banking invariant

Implement an `Account` with:

- read-only `balance`;
- read/write `overdraft_limit`;
- `deposit(amount)` and `withdraw(amount)` methods;
- invariant `balance >= -overdraft_limit`;
- rejection of a new overdraft limit that would invalidate the current balance.

This demonstrates when a property is appropriate versus an explicit domain method.

In [9]:
# Solution 9

class Account:
    def __init__(self, opening_balance=0.0, overdraft_limit=0.0):
        self._balance = float(opening_balance)
        self._overdraft_limit = 0.0
        self.overdraft_limit = overdraft_limit

        if self._balance < -self._overdraft_limit:
            raise ValueError("opening balance exceeds overdraft limit")

    @property
    def balance(self):
        return self._balance

    @property
    def overdraft_limit(self):
        return self._overdraft_limit

    @overdraft_limit.setter
    def overdraft_limit(self, value):
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise TypeError("overdraft_limit must be numeric")
        if value < 0:
            raise ValueError("overdraft_limit must be non-negative")
        if hasattr(self, "_balance") and self._balance < -float(value):
            raise ValueError("new overdraft limit is too small for current balance")
        self._overdraft_limit = float(value)

    @staticmethod
    def _positive_amount(amount):
        if isinstance(amount, bool) or not isinstance(amount, (int, float)):
            raise TypeError("amount must be numeric")
        if amount <= 0:
            raise ValueError("amount must be positive")
        return float(amount)

    def deposit(self, amount):
        self._balance += self._positive_amount(amount)

    def withdraw(self, amount):
        amount = self._positive_amount(amount)
        new_balance = self._balance - amount
        if new_balance < -self._overdraft_limit:
            raise ValueError("withdrawal exceeds overdraft limit")
        self._balance = new_balance


acct = Account(100, overdraft_limit=50)
acct.withdraw(130)
assert acct.balance == -30

try:
    acct.overdraft_limit = 20
except ValueError as exc:
    print("expected:", exc)

acct.deposit(40)
acct.overdraft_limit = 20
assert acct.balance == 10
assert acct.overdraft_limit == 20
print(acct.balance, acct.overdraft_limit)

expected: new overdraft limit is too small for current balance
10.0 20.0


## Problem 10 — Percentage with canonical storage

Implement `Discount.percent` in `[0, 100]`, but internally store `_rate` in `[0.0, 1.0]`. Add a read-only `rate` and an `apply(price)` method.

In [10]:
# Solution 10

class Discount:
    def __init__(self, percent):
        self.percent = percent

    @property
    def percent(self):
        return self._rate * 100

    @percent.setter
    def percent(self, value):
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise TypeError("percent must be numeric")
        if not 0 <= value <= 100:
            raise ValueError("percent must be between 0 and 100")
        self._rate = float(value) / 100

    @property
    def rate(self):
        return self._rate

    def apply(self, price):
        return price * (1 - self._rate)


discount = Discount(25)
assert discount.percent == 25
assert discount.rate == 0.25
assert discount.apply(200) == 150

discount.percent = 12.5
assert discount.rate == 0.125
print(discount.percent, discount.rate, discount.apply(80))

12.5 0.125 70.0


# Part IV — Deleters and Write-Only Properties

## Problem 11 — Property deleter with meaningful semantics

Build `ApiCredential.token` as a readable/writable/deletable property. After deletion, reading must raise `AttributeError`. Add read-only `has_token`.

In [11]:
# Solution 11

class ApiCredential:
    def __init__(self, token):
        self.token = token

    @property
    def token(self):
        if not hasattr(self, "_token"):
            raise AttributeError("token has been deleted")
        return self._token

    @token.setter
    def token(self, value):
        if not isinstance(value, str):
            raise TypeError("token must be a string")
        if not value.strip():
            raise ValueError("token cannot be empty")
        self._token = value

    @token.deleter
    def token(self):
        if hasattr(self, "_token"):
            del self._token

    @property
    def has_token(self):
        return hasattr(self, "_token")


cred = ApiCredential("abc123")
assert cred.has_token
assert cred.token == "abc123"

del cred.token
assert not cred.has_token

try:
    _ = cred.token
except AttributeError as exc:
    print("expected:", exc)

cred.token = "new-token"
assert cred.has_token
print(cred.token)

expected: token has been deleted
new-token


## Problem 12 — Write-only secret, with the privacy caveat

Create `SecretBox.secret` as a write-only property. Assignment stores a SHA-256 digest, reading raises `AttributeError`, and `verify(candidate)` compares a candidate.

A write-only property is not a security boundary; this design avoids storing the raw secret.

In [12]:
# Solution 12

import hashlib
import hmac

class SecretBox:
    secret = property(doc="Write-only secret; stores only a SHA-256 digest.")

    def __init__(self, secret):
        self.secret = secret

    @secret.setter
    def secret(self, value):
        if not isinstance(value, str):
            raise TypeError("secret must be a string")
        if len(value) < 8:
            raise ValueError("secret must contain at least 8 characters")
        self._secret_digest = hashlib.sha256(value.encode("utf-8")).digest()

    def verify(self, candidate):
        if not isinstance(candidate, str):
            return False
        candidate_digest = hashlib.sha256(candidate.encode("utf-8")).digest()
        return hmac.compare_digest(self._secret_digest, candidate_digest)


box = SecretBox("correct-horse")
assert box.verify("correct-horse")
assert not box.verify("wrong-secret")

try:
    _ = box.secret
except AttributeError as exc:
    print("secret is unreadable:", exc)

print(SecretBox.secret.__doc__)

secret is unreadable: property 'secret' of 'SecretBox' object has no getter
Write-only secret; stores only a SHA-256 digest.


# Part V — Property Documentation and Introspection

## Problem 13 — Inspect `fget`, `fset`, `fdel`, and `__doc__`

Create a fully managed `label` property and verify all accessors plus the getter-derived property docstring.

In [13]:
# Solution 13

class LabeledThing:
    def __init__(self, label):
        self.label = label

    @property
    def label(self):
        """Human-readable label for this object."""
        return self._label

    @label.setter
    def label(self, value):
        if not isinstance(value, str) or not value.strip():
            raise ValueError("label must be a non-empty string")
        self._label = value.strip()

    @label.deleter
    def label(self):
        del self._label


prop = LabeledThing.__dict__["label"]

assert isinstance(prop, property)
assert prop.fget is not None
assert prop.fset is not None
assert prop.fdel is not None
assert prop.__doc__ == "Human-readable label for this object."

print("getter:", prop.fget.__name__)
print("setter:", prop.fset.__name__)
print("deleter:", prop.fdel.__name__)
print("doc:", prop.__doc__)

getter: label
setter: label
deleter: label
doc: Human-readable label for this object.


## Problem 14 — Setter docstring does not become the property docstring

Create a getter with no docstring and a setter with a docstring. Inspect both `Class.attr.__doc__` and `Class.attr.fset.__doc__`.

In [14]:
# Solution 14

class SetterDocDemo:
    def __init__(self, value):
        self._value = value

    @property
    def value(self):
        return self._value

    @value.setter
    def value(self, new_value):
        """This setter docstring does not become the property docstring."""
        self._value = new_value


assert SetterDocDemo.value.__doc__ is None
assert SetterDocDemo.value.fset.__doc__ is not None

print("property docstring:", SetterDocDemo.value.__doc__)
print("setter docstring:", SetterDocDemo.value.fset.__doc__)

property docstring: None
setter docstring: This setter docstring does not become the property docstring.


# Part VI — Cached Computation with Invalidation

## Problem 15 — Cached derived property

Build `Circle.radius` and a read-only `area` that caches after first access. Changing the radius must invalidate the cache. Track computation count.

In [15]:
# Solution 15

import math

class Circle:
    def __init__(self, radius):
        self._area_computations = 0
        self.radius = radius

    @property
    def radius(self):
        return self._radius

    @radius.setter
    def radius(self, value):
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise TypeError("radius must be numeric")
        if value <= 0:
            raise ValueError("radius must be positive")

        self._radius = float(value)

        if hasattr(self, "_cached_area"):
            del self._cached_area

    @property
    def area(self):
        if not hasattr(self, "_cached_area"):
            self._area_computations += 1
            self._cached_area = math.pi * self._radius ** 2
        return self._cached_area

    @property
    def area_computations(self):
        return self._area_computations


circle = Circle(2)

a1 = circle.area
a2 = circle.area
a3 = circle.area
assert a1 == a2 == a3
assert circle.area_computations == 1

circle.radius = 3
_ = circle.area
assert circle.area_computations == 2

print("area:", circle.area)
print("computations:", circle.area_computations)

area: 28.274333882308138
computations: 2


### Best-practice note

Manual caching is appropriate only when the computed value is expensive enough to justify cache complexity. For cheap calculations, recomputing is usually clearer.

# Part VII — Inheritance and Property Overrides

## Problem 16 — Override only the setter in a subclass

Create `Product.price` requiring `price > 0`, then `DiscountProduct.price` allowing `price >= 0` while reusing the base getter with `@Product.price.setter`.

In [16]:
# Solution 16

class Product:
    def __init__(self, price):
        self.price = price

    @property
    def price(self):
        """Unit price."""
        return self._price

    @price.setter
    def price(self, value):
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise TypeError("price must be numeric")
        if value <= 0:
            raise ValueError("price must be > 0")
        self._price = float(value)


class DiscountProduct(Product):
    @Product.price.setter
    def price(self, value):
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise TypeError("price must be numeric")
        if value < 0:
            raise ValueError("price must be >= 0")
        self._price = float(value)


p = Product(10)
d = DiscountProduct(10)

try:
    p.price = 0
except ValueError:
    pass
else:
    raise AssertionError("Product should reject zero")

d.price = 0
assert d.price == 0.0

assert Product.price.fget is DiscountProduct.price.fget
assert Product.price.fset is not DiscountProduct.price.fset

print("base price:", p.price)
print("discount price:", d.price)

base price: 10.0
discount price: 0.0


## Problem 17 — Override only the getter in a subclass

Create `Person.name` with a normal getter/setter. In `MaskedPerson`, reuse the base setter but override the getter so `"Ada Lovelace"` displays as `"A. L."`.

In [17]:
# Solution 17

class Person:
    def __init__(self, name):
        self.name = name

    @property
    def name(self):
        return self._name

    @name.setter
    def name(self, value):
        if not isinstance(value, str) or not value.strip():
            raise ValueError("name must be a non-empty string")
        self._name = value.strip()


class MaskedPerson(Person):
    @Person.name.getter
    def name(self):
        parts = self._name.split()
        return " ".join(part[0].upper() + "." for part in parts)


person = MaskedPerson("Ada Lovelace")
assert person.name == "A. L."

person.name = "grace hopper"
assert person.name == "G. H."
assert MaskedPerson.name.fset is Person.name.fset

print(person.name)

G. H.


# Part VIII — Reusable Property Factories

## Problem 18 — Build a reusable validated-property factory

Write `bounded_number(name, minimum, maximum)` that returns a `property`. Use it for `RGB.red`, `green`, and `blue`, each constrained to integer `[0, 255]`.

In [18]:
# Solution 18

def bounded_number(name, minimum, maximum):
    private_name = "_" + name

    def getter(self):
        return getattr(self, private_name)

    def setter(self, value):
        if isinstance(value, bool) or not isinstance(value, int):
            raise TypeError(f"{name} must be an integer")
        if not minimum <= value <= maximum:
            raise ValueError(f"{name} must be in [{minimum}, {maximum}]")
        setattr(self, private_name, value)

    return property(
        getter,
        setter,
        doc=f"{name} value constrained to [{minimum}, {maximum}].",
    )


class RGB:
    red = bounded_number("red", 0, 255)
    green = bounded_number("green", 0, 255)
    blue = bounded_number("blue", 0, 255)

    def __init__(self, red, green, blue):
        self.red = red
        self.green = green
        self.blue = blue

    @property
    def hex(self):
        return f"#{self.red:02X}{self.green:02X}{self.blue:02X}"


color = RGB(255, 127, 0)
assert color.hex == "#FF7F00"
assert RGB.red.__doc__ == "red value constrained to [0, 255]."

try:
    color.blue = 300
except ValueError as exc:
    print("expected:", exc)

print(color.red, color.green, color.blue, color.hex)

expected: blue must be in [0, 255]
255 127 0 #FF7F00


### Design note

A factory can reduce repetition, but excessive abstraction can make simple models harder to read. Prefer explicit properties until repeated validation logic is stable and reusable.

# Part IX — `__slots__` and Properties

## Problem 19 — Properties with `__slots__`

Create `Point` with `__slots__ = ("_x", "_y")`, validated `x` and `y`, and read-only `magnitude`. Verify instances have no `__dict__`.

In [19]:
# Solution 19

import math

class Point:
    __slots__ = ("_x", "_y")

    def __init__(self, x, y):
        self.x = x
        self.y = y

    @staticmethod
    def _validate_coordinate(value):
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise TypeError("coordinate must be numeric")
        return float(value)

    @property
    def x(self):
        return self._x

    @x.setter
    def x(self, value):
        self._x = self._validate_coordinate(value)

    @property
    def y(self):
        return self._y

    @y.setter
    def y(self, value):
        self._y = self._validate_coordinate(value)

    @property
    def magnitude(self):
        return math.hypot(self._x, self._y)


pt = Point(3, 4)
assert pt.magnitude == 5.0
assert not hasattr(pt, "__dict__")

pt.x = 6
assert pt.x == 6.0

print(pt.x, pt.y, pt.magnitude)
print("has __dict__:", hasattr(pt, "__dict__"))

6.0 4.0 7.211102550927978
has __dict__: False


# Part X — API Design: Property or Method?

## Problem 20 — Choose properties and methods deliberately

Implement an `Order`.

Use properties for cheap attribute-like facts (`subtotal`, `tax_rate`, `tax`, `total`) and methods for domain operations (`add_item`, `remove_item`).

In [20]:
# Solution 20

class Order:
    def __init__(self, tax_rate=0.0):
        self._items = {}
        self.tax_rate = tax_rate

    @property
    def tax_rate(self):
        return self._tax_rate

    @tax_rate.setter
    def tax_rate(self, value):
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise TypeError("tax_rate must be numeric")
        if not 0 <= value <= 1:
            raise ValueError("tax_rate must be between 0 and 1")
        self._tax_rate = float(value)

    def add_item(self, name, unit_price, quantity=1):
        if not isinstance(name, str) or not name.strip():
            raise ValueError("name must be a non-empty string")
        if isinstance(unit_price, bool) or not isinstance(unit_price, (int, float)):
            raise TypeError("unit_price must be numeric")
        if unit_price < 0:
            raise ValueError("unit_price must be non-negative")
        if isinstance(quantity, bool) or not isinstance(quantity, int):
            raise TypeError("quantity must be an integer")
        if quantity <= 0:
            raise ValueError("quantity must be positive")

        self._items[name] = (float(unit_price), quantity)

    def remove_item(self, name):
        del self._items[name]

    @property
    def subtotal(self):
        return sum(price * qty for price, qty in self._items.values())

    @property
    def tax(self):
        return self.subtotal * self._tax_rate

    @property
    def total(self):
        return self.subtotal + self.tax


order = Order(tax_rate=0.20)
order.add_item("Book", 25, 2)
order.add_item("Pen", 2, 5)

assert order.subtotal == 60
assert order.tax == 12
assert order.total == 72

order.tax_rate = 0.10
assert order.total == 66

print(order.subtotal, order.tax, order.total)

60.0 6.0 66.0


# Part XI — Debugging Problems

## Problem 21 — Fix a validation bypass

A constructor writes directly to `_age`, bypassing the `age` setter. Fix the class so initialization and later assignment use the same validation path.

In [21]:
# Solution 21

class Age:
    def __init__(self, age):
        self.age = age

    @property
    def age(self):
        return self._age

    @age.setter
    def age(self, value):
        if isinstance(value, bool) or not isinstance(value, int):
            raise TypeError("age must be an integer")
        if not 0 <= value <= 130:
            raise ValueError("age must be between 0 and 130")
        self._age = value


a = Age(36)
assert a.age == 36

for bad in (-1, 131):
    try:
        Age(bad)
    except ValueError:
        pass
    else:
        raise AssertionError("constructor should enforce the property invariant")

print(a.age)

36


## Problem 22 — Preserve old state on failed assignment

Implement `EmailAddress.email` so all validation occurs before mutation. A failed assignment must leave the previous valid email unchanged.

In [22]:
# Solution 22

class EmailAddress:
    def __init__(self, email):
        self.email = email

    @property
    def email(self):
        return self._email

    @email.setter
    def email(self, value):
        if not isinstance(value, str):
            raise TypeError("email must be a string")

        candidate = value.strip()

        if candidate.count("@") != 1:
            raise ValueError("email must contain exactly one @")
        local, domain = candidate.split("@")
        if not local or "." not in domain or domain.startswith(".") or domain.endswith("."):
            raise ValueError("invalid email structure")

        self._email = candidate


e = EmailAddress("ada@example.com")

try:
    e.email = "not-an-email"
except ValueError:
    pass

assert e.email == "ada@example.com"
print(e.email)

ada@example.com


## Problem 23 — Read-only does not mean immutable internals

First return an internal list through a getter and show that a caller can mutate it. Then fix the design by returning an immutable tuple snapshot.

In [23]:
# Solution 23

class UnsafeTagCollection:
    def __init__(self, tags):
        self._tags = list(tags)

    @property
    def tags(self):
        return self._tags


unsafe = UnsafeTagCollection(["python", "oop"])
unsafe.tags.append("mutated-from-outside")
assert "mutated-from-outside" in unsafe.tags


class SafeTagCollection:
    def __init__(self, tags):
        self._tags = list(tags)

    @property
    def tags(self):
        return tuple(self._tags)

    def add_tag(self, tag):
        if not isinstance(tag, str) or not tag:
            raise ValueError("tag must be a non-empty string")
        self._tags.append(tag)


safe = SafeTagCollection(["python", "oop"])
snapshot = safe.tags
assert isinstance(snapshot, tuple)

try:
    snapshot.append("x")
except AttributeError as exc:
    print("tuple snapshot cannot be appended to:", exc)

safe.add_tag("properties")
assert safe.tags == ("python", "oop", "properties")
print(safe.tags)

tuple snapshot cannot be appended to: 'tuple' object has no attribute 'append'
('python', 'oop', 'properties')


# Part XII — State Transitions and Immutability Patterns

## Problem 24 — Write-once property

Implement `Entity.id` so it may be assigned exactly once. The constructor assigns it; later assignment raises `AttributeError`.

In [24]:
# Solution 24

class Entity:
    def __init__(self, entity_id):
        self.id = entity_id

    @property
    def id(self):
        return self._id

    @id.setter
    def id(self, value):
        if hasattr(self, "_id"):
            raise AttributeError("id is write-once")
        if not isinstance(value, str) or not value.strip():
            raise ValueError("id must be a non-empty string")
        self._id = value.strip()


entity = Entity("A-100")
assert entity.id == "A-100"

try:
    entity.id = "B-200"
except AttributeError as exc:
    print("expected:", exc)

assert entity.id == "A-100"

expected: id is write-once


## Problem 25 — Controlled state transitions

Implement `Job.status` with transitions:

- `pending -> running`
- `running -> succeeded`
- `running -> failed`
- terminal states cannot transition further

In [25]:
# Solution 25

class Job:
    _TRANSITIONS = {
        "pending": {"running"},
        "running": {"succeeded", "failed"},
        "succeeded": set(),
        "failed": set(),
    }

    def __init__(self):
        self._status = "pending"

    @property
    def status(self):
        return self._status

    @status.setter
    def status(self, new_status):
        if new_status not in self._TRANSITIONS:
            raise ValueError(f"unknown status: {new_status!r}")

        allowed = self._TRANSITIONS[self._status]
        if new_status not in allowed:
            raise ValueError(
                f"cannot transition from {self._status!r} to {new_status!r}"
            )

        self._status = new_status


job = Job()
job.status = "running"
job.status = "succeeded"
assert job.status == "succeeded"

try:
    job.status = "running"
except ValueError as exc:
    print("expected:", exc)

print(job.status)

expected: cannot transition from 'succeeded' to 'running'
succeeded


# Part XIII — Capstone

## Problem 26 — Inventory item with multiple cooperating properties

Build `InventoryItem` with:

### Managed properties
- `sku`: normalized uppercase, non-empty, write-once.
- `unit_price`: non-negative number.
- `quantity`: non-negative integer.
- `discount_percent`: number in `[0, 100]`.

### Derived read-only properties
- `gross_value`
- `net_unit_price`
- `net_value`

### Deleter behavior
- `del item.discount_percent` resets discount to `0.0`.

### Methods
- `restock(amount)`
- `sell(amount)`

### Best-practice constraints
- route initialization through public validation where practical;
- do not corrupt previous valid state after failed assignments;
- reject `bool` where a numeric value is expected;
- document public properties in getter docstrings.

In [26]:
# Solution 26

class InventoryItem:
    def __init__(self, sku, unit_price, quantity=0, discount_percent=0):
        self.sku = sku
        self.unit_price = unit_price
        self.quantity = quantity
        self.discount_percent = discount_percent

    @property
    def sku(self):
        """Normalized, write-once stock-keeping unit."""
        return self._sku

    @sku.setter
    def sku(self, value):
        if hasattr(self, "_sku"):
            raise AttributeError("sku is write-once")
        if not isinstance(value, str):
            raise TypeError("sku must be a string")
        normalized = value.strip().upper()
        if not normalized:
            raise ValueError("sku cannot be empty")
        self._sku = normalized

    @property
    def unit_price(self):
        """Non-negative unit price."""
        return self._unit_price

    @unit_price.setter
    def unit_price(self, value):
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise TypeError("unit_price must be numeric")
        if value < 0:
            raise ValueError("unit_price must be non-negative")
        self._unit_price = float(value)

    @property
    def quantity(self):
        """Current non-negative inventory quantity."""
        return self._quantity

    @quantity.setter
    def quantity(self, value):
        if isinstance(value, bool) or not isinstance(value, int):
            raise TypeError("quantity must be an integer")
        if value < 0:
            raise ValueError("quantity must be non-negative")
        self._quantity = value

    @property
    def discount_percent(self):
        """Discount percentage in the inclusive range [0, 100]."""
        return self._discount_percent

    @discount_percent.setter
    def discount_percent(self, value):
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise TypeError("discount_percent must be numeric")
        if not 0 <= value <= 100:
            raise ValueError("discount_percent must be between 0 and 100")
        self._discount_percent = float(value)

    @discount_percent.deleter
    def discount_percent(self):
        self._discount_percent = 0.0

    @property
    def gross_value(self):
        """Inventory value before discount."""
        return self._unit_price * self._quantity

    @property
    def net_unit_price(self):
        """Unit price after discount."""
        return self._unit_price * (1 - self._discount_percent / 100)

    @property
    def net_value(self):
        """Inventory value after discount."""
        return self.net_unit_price * self._quantity

    def restock(self, amount):
        if isinstance(amount, bool) or not isinstance(amount, int):
            raise TypeError("restock amount must be an integer")
        if amount <= 0:
            raise ValueError("restock amount must be positive")
        self.quantity = self._quantity + amount

    def sell(self, amount):
        if isinstance(amount, bool) or not isinstance(amount, int):
            raise TypeError("sell amount must be an integer")
        if amount <= 0:
            raise ValueError("sell amount must be positive")
        if amount > self._quantity:
            raise ValueError("not enough stock")
        self.quantity = self._quantity - amount


item = InventoryItem("  py-001  ", unit_price=50, quantity=10, discount_percent=20)

assert item.sku == "PY-001"
assert item.gross_value == 500
assert item.net_unit_price == 40
assert item.net_value == 400

item.sell(3)
assert item.quantity == 7
assert item.net_value == 280

item.restock(5)
assert item.quantity == 12

old_price = item.unit_price
try:
    item.unit_price = -1
except ValueError:
    pass
assert item.unit_price == old_price

del item.discount_percent
assert item.discount_percent == 0
assert item.net_value == item.gross_value

try:
    item.sku = "NEW-SKU"
except AttributeError as exc:
    print("expected:", exc)

print({
    "sku": item.sku,
    "unit_price": item.unit_price,
    "quantity": item.quantity,
    "discount_percent": item.discount_percent,
    "gross_value": item.gross_value,
    "net_value": item.net_value,
})

expected: sku is write-once
{'sku': 'PY-001', 'unit_price': 50.0, 'quantity': 12, 'discount_percent': 0.0, 'gross_value': 600.0, 'net_value': 600.0}


# Part XIV — Extra Challenge Set

The next exercises add more lines of code and extra practice.

### Problem 27 — Reject invalid volume
Implement `Volume.level` in `[0, 100]`. Reject out-of-range values instead of silently clamping them.

### Problem 28 — Multiple public units
Store a distance internally in meters. Expose read/write `kilometers` and `miles`.

### Problem 29 — Two-way dependent validation
Create `ExamScore(points, maximum)` where `0 <= points <= maximum` and `maximum > 0`.

### Problem 30 — Lazy parse with invalidation
Store a raw ISO date string and expose a cached read-only parsed `datetime.date`.

### Problem 31 — Subclass restriction
A base vehicle accepts any non-negative speed; a subclass caps it at `120`.

### Problem 32 — Property factory with text normalization
Write `normalized_text(name)` returning a property that strips and collapses whitespace.

### Problem 33 — Safe mapping exposure
Expose an internal mapping through `types.MappingProxyType`.

### Problem 34 — Delete to reset
Deleting `Preferences.theme` restores `"system"`.

### Problem 35 — Audit successful setter calls
Count successful setter changes, but do not increment the counter when validation fails.

In [27]:
# Solution 27

class Volume:
    def __init__(self, level):
        self.level = level

    @property
    def level(self):
        return self._level

    @level.setter
    def level(self, value):
        if isinstance(value, bool) or not isinstance(value, int):
            raise TypeError("level must be an integer")
        if not 0 <= value <= 100:
            raise ValueError("level must be between 0 and 100")
        self._level = value


v = Volume(50)
try:
    v.level = 150
except ValueError:
    pass
assert v.level == 50

In [28]:
# Solution 28

class Distance:
    MILES_PER_KM = 0.621371192237334

    def __init__(self, meters):
        self._meters = float(meters)

    @property
    def meters(self):
        return self._meters

    @property
    def kilometers(self):
        return self._meters / 1000

    @kilometers.setter
    def kilometers(self, value):
        if value < 0:
            raise ValueError("distance cannot be negative")
        self._meters = float(value) * 1000

    @property
    def miles(self):
        return self.kilometers * self.MILES_PER_KM

    @miles.setter
    def miles(self, value):
        if value < 0:
            raise ValueError("distance cannot be negative")
        self._meters = float(value) / self.MILES_PER_KM * 1000


d = Distance(1000)
assert abs(d.kilometers - 1) < 1e-12
d.miles = 1
assert abs(d.miles - 1) < 1e-12

In [29]:
# Solution 29

class ExamScore:
    def __init__(self, points, maximum):
        if maximum <= 0:
            raise ValueError("maximum must be positive")
        if not 0 <= points <= maximum:
            raise ValueError("points must satisfy 0 <= points <= maximum")
        self._points = points
        self._maximum = maximum

    @property
    def points(self):
        return self._points

    @points.setter
    def points(self, value):
        if not 0 <= value <= self._maximum:
            raise ValueError("points out of range")
        self._points = value

    @property
    def maximum(self):
        return self._maximum

    @maximum.setter
    def maximum(self, value):
        if value <= 0:
            raise ValueError("maximum must be positive")
        if value < self._points:
            raise ValueError("maximum cannot be lower than current points")
        self._maximum = value

    @property
    def ratio(self):
        return self._points / self._maximum


score = ExamScore(80, 100)
score.maximum = 120
score.points = 90
assert score.ratio == 0.75

In [30]:
# Solution 30

from datetime import date

class DateText:
    def __init__(self, raw):
        self.raw = raw

    @property
    def raw(self):
        return self._raw

    @raw.setter
    def raw(self, value):
        if not isinstance(value, str):
            raise TypeError("raw date must be a string")

        date.fromisoformat(value)
        self._raw = value

        if hasattr(self, "_parsed"):
            del self._parsed

    @property
    def parsed(self):
        if not hasattr(self, "_parsed"):
            self._parsed = date.fromisoformat(self._raw)
        return self._parsed


dt = DateText("2026-09-11")
first = dt.parsed
second = dt.parsed
assert first is second

dt.raw = "2027-01-01"
assert dt.parsed == date(2027, 1, 1)

In [31]:
# Solution 31

class Vehicle:
    def __init__(self, speed=0):
        self.speed = speed

    @property
    def speed(self):
        return self._speed

    @speed.setter
    def speed(self, value):
        if value < 0:
            raise ValueError("speed cannot be negative")
        self._speed = float(value)


class CityVehicle(Vehicle):
    @Vehicle.speed.setter
    def speed(self, value):
        if not 0 <= value <= 120:
            raise ValueError("city vehicle speed must be in [0, 120]")
        self._speed = float(value)


car = CityVehicle(50)
car.speed = 120
assert car.speed == 120

try:
    car.speed = 121
except ValueError:
    pass

In [32]:
# Solution 32

import re

def normalized_text(name):
    private_name = "_" + name

    def getter(self):
        return getattr(self, private_name)

    def setter(self, value):
        if not isinstance(value, str):
            raise TypeError(f"{name} must be a string")
        normalized = re.sub(r"\s+", " ", value).strip()
        if not normalized:
            raise ValueError(f"{name} cannot be empty")
        setattr(self, private_name, normalized)

    return property(getter, setter)


class Article:
    title = normalized_text("title")
    author = normalized_text("author")

    def __init__(self, title, author):
        self.title = title
        self.author = author


article = Article("  Advanced   Python  ", "  Ada   Lovelace ")
assert article.title == "Advanced Python"
assert article.author == "Ada Lovelace"

In [33]:
# Solution 33

from types import MappingProxyType

class Metrics:
    def __init__(self):
        self._values = {}

    @property
    def values(self):
        return MappingProxyType(self._values)

    def record(self, key, value):
        self._values[key] = value


metrics = Metrics()
metrics.record("requests", 10)

view = metrics.values
assert view["requests"] == 10

try:
    view["requests"] = 999
except TypeError as exc:
    print("read-only mapping view:", exc)

read-only mapping view: 'mappingproxy' object does not support item assignment


In [34]:
# Solution 34

class Preferences:
    def __init__(self, theme="system"):
        self.theme = theme

    @property
    def theme(self):
        return self._theme

    @theme.setter
    def theme(self, value):
        if value not in {"system", "light", "dark"}:
            raise ValueError("theme must be system, light, or dark")
        self._theme = value

    @theme.deleter
    def theme(self):
        self._theme = "system"


prefs = Preferences("dark")
del prefs.theme
assert prefs.theme == "system"

In [35]:
# Solution 35

class AuditedValue:
    def __init__(self, value):
        self._successful_changes = 0
        self.value = value

    @property
    def value(self):
        return self._value

    @value.setter
    def value(self, value):
        if isinstance(value, bool) or not isinstance(value, int):
            raise TypeError("value must be an integer")
        if value < 0:
            raise ValueError("value must be non-negative")

        self._value = value
        self._successful_changes += 1

    @property
    def successful_changes(self):
        return self._successful_changes


audit = AuditedValue(1)
assert audit.successful_changes == 1

audit.value = 2
assert audit.successful_changes == 2

try:
    audit.value = -1
except ValueError:
    pass

assert audit.value == 2
assert audit.successful_changes == 2

# Final Review — Best Practices Checklist

- Use a separate backing name such as `_name`; otherwise setters can recurse.
- Route constructor assignment through the public property when you want one shared validation path.
- Validate completely **before** mutating state.
- Raise `TypeError` for inappropriate types and `ValueError` for inappropriate values.
- Remember that `bool` is a subclass of `int`; reject it explicitly when booleans should not count as numbers.
- Keep cheap, deterministic, attribute-like calculations as read-only properties.
- Prefer explicit methods for operations with side effects or important domain meaning.
- Keep getter/setter/deleter function names aligned with the property name when using decorator syntax.
- Put user-facing property documentation in the getter docstring; for a property without a getter, pass `doc=...` to `property(...)`.
- Remember `.setter`, `.getter`, and `.deleter` return new property objects.
- A getter-only property prevents direct assignment through that property, but it does not make referenced mutable objects immutable.
- A write-only property is not a security boundary.
- Use inheritance overrides carefully: `@BaseClass.attr.setter` and `@BaseClass.attr.getter` preserve the other accessor.
- Cache derived values only when the computation is expensive enough to justify invalidation logic.
- Property factories are useful for stable repetitive rules, but explicit code is often easier to maintain.

# Suggested Mastery Tasks

Without looking at the solutions, rebuild these from scratch:

1. `Temperature`
2. `Interval`
3. `ApiCredential`
4. `Circle` with cache invalidation
5. `DiscountProduct` using an inherited setter
6. `RGB` using a property factory
7. `InventoryItem` capstone

Then write additional tests for invalid types, boundary values, failed updates, deletion, and subclass behavior.